# Adaptive Recovery Algorithm 
**March 2026**
- Extend to 2 qubits system

**April 2026**
Here we implement the updated adaptive recovery protocol, where the system's state collapses due to the measurement outcome on the ancilla.

## Fixed Choices
- `N qubits`: 1
- `Noises`: Amplitude Damping (50\%) / Bitflip (50\%)



In [241]:
using Pkg
Pkg.activate("../julia")
Pkg.instantiate()

  Activating project at `~/UNIPA/COLLISION_MODELS/shooting-decoherences/julia`


In [242]:
using LinearAlgebra
using Plots
using Revise
using JSON
using Latexify

using Random
using StatsBase

using Printf

# ==============================================
# Custom logger
using Logging, TerminalLoggers, ProgressLogging
# Display debug messages
global_logger(TerminalLogger(stderr, Logging.Info))
debuglogger = ConsoleLogger(stderr, Logging.Debug)

includet("../julia/src/logging.jl")
# ==============================================

includet("../julia/src/UnitaryDilation/UnitaryDilation.jl")
includet("../julia/src/PetzMaps.jl")
# includet("../julia/src/DecoKiller.jl")
includet("../julia/src/utils.jl")
includet("../julia/src/configurations.jl")
includet("../julia/src/custom_plots.jl")
includet("../julia/src/quantum_states.jl")

using .UnitaryDilation
# using .DecoKiller
using .PetzMaps

In [243]:
function apply_noise(model, ρ, n_qubits)
  ρf = apply_channel(model.kraus_fwd, ρ, n_qubits)
  # Enforce physicality (hermitianicity and trace 1)
  # enforce_physical!(ρf)
  return ρf
end


function recovery(model, ρ)
  ρr, η = apply_collision(model, ρ)
  # enforce_physical!(ρr)
  return ρr, η
end

recovery (generic function with 1 method)

## Setup

In [244]:
n_qubits = 1
beta = 1.0
dt = π/10
gamma = -log(0.5) / dt
n_steps = 2

recovery_type = "auto"  # 'auto', 'random', 'codespace', 'inputspace'

fidelities = Float64[]
fidelities_ref = Float64[]  # Store the fidelities of the noisy state evolving without recovery

# Random but reproducible states
seed = 42
rng = Xoshiro(seed)

Xoshiro(0xa379de7eeeb2a4e8, 0x953dccb6b532b3af, 0xf597b8ff8cfd652a, 0xccd7337c571680d1, 0xc90c4a0730db3f7e)

### Initial States
We create the recovery state `sigma` and a random initial state

In [245]:
# Choose a reference state for the recovery
rx = 1 / sqrt(5)
ry = 1 / sqrt(5)
rz = 1 / sqrt(2)
target = 0.5 * [
  [1 + rz         rx - im * ry]; 
  [rx + ry * im   1 - rz      ]
  ]

sigma = target
ρ0 = copy(sigma)

display(latexify(sigma; fmt="%.3f")) # display as LaTeX table
display(latexify(ρ0; fmt="%.3f"))

L"\begin{equation}
\left[
\begin{array}{cc}
0.854+0.000\mathit{i} & 0.224-0.224\mathit{i} \\
0.224+0.224\mathit{i} & 0.146+0.000\mathit{i} \\
\end{array}
\right]
\end{equation}
"

L"\begin{equation}
\left[
\begin{array}{cc}
0.854+0.000\mathit{i} & 0.224-0.224\mathit{i} \\
0.224+0.224\mathit{i} & 0.146+0.000\mathit{i} \\
\end{array}
\right]
\end{equation}
"

### Noise

In [246]:
noise_probabilities = [
  (0.50, "amplitude_damping"),
  (0.50, "bitflip"),
]

# ======================================
# Precompute the supermaps once
noise_options = [
  NoiseObj(noise_model[2], noise_model[1], sigma, gamma, dt)
  for noise_model in noise_probabilities
]

# ======================================
# Fix the real Noise model
noise = "amplitude_damping"
println("Chosen noise:\t\t$noise")
real_noise_idx = findfirst(n -> n.name == noise, noise_options)
real_noise = noise_options[real_noise_idx]
# Take the relevant Kraus operators
real_kraus = get_kraus_operators(noise, gamma, dt)
real_model = CollisionModel(real_kraus, sigma, n=n_qubits)

Chosen noise:		amplitude_damping


CollisionModel{ComplexF64}(2, 2, ComplexF64[0.8535533905932737 + 0.0im 0.22360679774997896 - 0.22360679774997896im; 0.22360679774997896 + 0.22360679774997896im 0.14644660940672627 + 0.0im], Matrix{ComplexF64}[[1.0 + 0.0im 0.0 + 0.0im; 0.0 + 0.0im 0.7071067811865476 + 0.0im], [0.0 + 0.0im 0.7071067811865476 + 0.0im; 0.0 + 0.0im 0.0 + 0.0im]], Matrix{ComplexF64}[[0.9228010679365007 - 8.326672684688674e-17im 0.04466229374383668 - 0.044662293743836234im; 0.07071919149012793 + 0.07071919149012787im 0.9133836040107832 + 5.551115123125783e-17im], [0.18949772235060225 - 0.1894977223506022im -8.326672684688674e-17 + 0.28967085721352925im; 0.25810270622979464 + 0.0im -0.1972710574969049 + 0.1972710574969049im]], ComplexF64[0.9228010679365007 - 8.326672684688674e-17im 0.0 + 0.0im 0.04466229374383668 - 0.044662293743836234im 0.0 + 0.0im; 0.18949772235060225 - 0.1894977223506022im 1.0 + 0.0im -8.326672684688674e-17 + 0.28967085721352925im 0.0 + 0.0im; 0.07071919149012793 + 0.07071919149012787im 0.0

In [247]:
display(kraus_to_superop(noise_options[1].kraus))
display(kraus_to_superop(noise_options[2].kraus))

4×4 Matrix{ComplexF64}:
 1.0+0.0im       0.0+0.0im       0.0+0.0im  0.5+0.0im
 0.0+0.0im  0.707107+0.0im       0.0+0.0im  0.0+0.0im
 0.0+0.0im       0.0+0.0im  0.707107+0.0im  0.0+0.0im
 0.0+0.0im       0.0+0.0im       0.0+0.0im  0.5+0.0im

4×4 Matrix{ComplexF64}:
 0.5+0.0im  0.0+0.0im  0.0+0.0im  0.5+0.0im
 0.0+0.0im  0.5+0.0im  0.5+0.0im  0.0+0.0im
 0.0+0.0im  0.5+0.0im  0.5+0.0im  0.0+0.0im
 0.5+0.0im  0.0+0.0im  0.0+0.0im  0.5+0.0im

In [248]:
model = CollisionModel(noise_options[1].kraus, sigma, n=n_qubits)
model.kraus_fwd
_, M_noise = build_superoperators(model)
M_noise

4×4 Matrix{ComplexF64}:
 1.0+0.0im       0.0+0.0im       0.0+0.0im  0.5+0.0im
 0.0+0.0im  0.707107+0.0im       0.0+0.0im  0.0+0.0im
 0.0+0.0im       0.0+0.0im  0.707107+0.0im  0.0+0.0im
 0.0+0.0im       0.0+0.0im       0.0+0.0im  0.5+0.0im

### Initialization

In [249]:

config = RecoveryConfig(
    "notebook", # name of the experiment
    "../experiments/example",
    sigma,      # reference state
    "auto",     # recovery type
    real_noise,
    n_qubits,
    n_steps,
    1,          # number of states
    seed,
    rng,
    dt,
    0.8
)

rho0, rho_to_rec, rho_free = copy(ρ0), copy(ρ0), copy(ρ0)

# Make a random choice (see configurations.jl for details)
choice = make_initial_choice(rng, noise_options)

noise_guess = deepcopy(noise_options[choice.current])
M_total = noise_guess.supermap_noise

println("Initial noise guess:\t$(noise_guess.name)")
println("Size of superoperator:\t$(size(M_total))")

state = RecoveryState(
    rho0, rho_to_rec, rho_free, noise_guess, M_total, choice, noise_options)

logs = RecoveryLogs()

Initial noise guess:	bitflip
Size of superoperator:	(4, 4)


RecoveryLogs(Float64[], Float64[], Int64[], @NamedTuple{Nx::Matrix{ComplexF64}, N1::Matrix{ComplexF64}, N2::Matrix{ComplexF64}, P::Matrix{ComplexF64}, Cx::Matrix{ComplexF64}, Xi::Matrix{ComplexF64}}[])

In [250]:
# Rename variables for readability
Ox = config.real_noise.supermap_noise
O1 = state.noise_options[1].supermap_noise
O2 = state.noise_options[2].supermap_noise
Nx = I(size(config.real_noise.supermap)[1])
N1 = I(size(config.real_noise.supermap)[1])
N2 = I(size(config.real_noise.supermap)[1])
;

In [251]:
display(N1)

4×4 Diagonal{Bool, Vector{Bool}}:
 1  ⋅  ⋅  ⋅
 ⋅  1  ⋅  ⋅
 ⋅  ⋅  1  ⋅
 ⋅  ⋅  ⋅  1

## Algorithm

### Step 1: Separate the states to track


In [252]:
rho_free = unvec(Nx * vec(state.ρ0))
rho_to_rec = copy(state.ρ0)
rho1 = copy(state.ρ0)
rho2 = copy(state.ρ0)
# rho_free = unvec(Nx * vec(state.ρ0))
# rho1 = copy(rho_to_rec)
# rho2 = copy(rho_to_rec)

println("Fidelity after noise: ", fidelity(state.ρ0, rho_to_rec))
display(latexify(rho_to_rec; fmt="%.3f"))

Fidelity after noise: 0.9999999999999964


L"\begin{equation}
\left[
\begin{array}{cc}
0.854+0.000\mathit{i} & 0.224-0.224\mathit{i} \\
0.224+0.224\mathit{i} & 0.146+0.000\mathit{i} \\
\end{array}
\right]
\end{equation}
"

### Step 2.1: Informative Collision
We let the state `rho_rec` (and `rho1`, `rho2` for tracking) interact with an ancilla with a Unitary SWAP interaction map,
$$
U_{\text{SWAP}}|a\rangle \otimes|b\rangle=|b\rangle\otimes|a\rangle
$$

Here we also start from a not-pure ancilla $\eta = p|0\rangle\langle 0| + (1-p)|1\rangle\langle 1|$

In [253]:
function embed_operator(op::Matrix, target_index::Int, n::Int)
    I2 = [1.0 0.0; 0.0 1.0] # 2x2 Identity
    
    # Start the Kronecker product chain
    result = (target_index == 1) ? op : I2
    for i in 2:n
        next_op = (i == target_index) ? op : I2
        result = kron(result, next_op)
    end
    return result
end

function n_qubit_exchange_unitary(n_qubits::Int, g::Float64=0.1, t::Float64=1.0)
  # Qubit raising and lowering operators
  sp = [0.0 1.0; 0.0 0.0]
  sm = [0.0 0.0; 1.0 0.0]

  # Total dimension, considering 1 qubit ancilla
  n_total = n_qubits + 1
  d = 2^(n_total)
  H_int = zeros(ComplexF64, d, d)

  # The ancilla is the 'last system' in the kronecker product
  ancilla_idx = n_total
  sp_anc = embed_operator(sp, ancilla_idx, n_total)
  sm_anc = embed_operator(sm, ancilla_idx, n_total)
    
  # Sum over all k system qubits
  for k in 1:n_qubits
      sp_k = embed_operator(sp, k, n_total)
      sm_k = embed_operator(sm, k, n_total)
      
      exchange_term = (sp_anc * sm_k) + (sm_anc * sp_k)
      
      H_int += (g / n_qubits) * exchange_term
  end
  
  # Return the time evolution unitary U(t)
  U = exp(-1im * H_int * t)
  return U
end
 

U = n_qubit_exchange_unitary(n_qubits, 1.0, dt)
# Check the collision
η = [[1.0 0.0]; [0.0 0.0]]

display(latexify(η))
display(latexify(ptrace_ancilla(U * kron(state.ρ0, η) * U', 2^n_qubits, 2); fmt="%0.3f"))
display(latexify(ptrace_sys(U * kron(state.ρ0, η) * U', 2^n_qubits, 2); fmt="%0.3f"))

L"\begin{equation}
\left[
\begin{array}{cc}
1.0 & 0.0 \\
0.0 & 0.0 \\
\end{array}
\right]
\end{equation}
"

L"\begin{equation}
\left[
\begin{array}{cc}
0.868+0.000\mathit{i} & 0.213-0.213\mathit{i} \\
0.213+0.213\mathit{i} & 0.132+0.000\mathit{i} \\
\end{array}
\right]
\end{equation}
"

L"\begin{equation}
\left[
\begin{array}{cc}
0.986+0.000\mathit{i} & 0.069+0.069\mathit{i} \\
0.069-0.069\mathit{i} & 0.014+0.000\mathit{i} \\
\end{array}
\right]
\end{equation}
"

The trail `_` in `rho_to_rec_`, `rho1_`, and `rho2_`, will indicate their entanglement with an ancilla.

In [254]:
kron(state.ρ0, η)

4×4 Matrix{ComplexF64}:
 0.853553+0.0im       0.0+0.0im  0.223607-0.223607im  0.0-0.0im
      0.0+0.0im       0.0+0.0im       0.0-0.0im       0.0-0.0im
 0.223607+0.223607im  0.0+0.0im  0.146447+0.0im       0.0+0.0im
      0.0+0.0im       0.0+0.0im       0.0+0.0im       0.0+0.0im

In [255]:
model = CollisionModel(
    U, sigma, 2^n_qubits, 2, ancilla_state=η)

# Get the composite state system+ancilla
rho_to_rec_ = apply_collision(model, rho_to_rec; ancilla_state=η, trace=false)
rho1_ = apply_collision(model, rho1; ancilla_state=η, trace=false)
rho2_ = apply_collision(model, rho2; ancilla_state=η, trace=false)

4×4 Matrix{ComplexF64}:
  0.853553+0.0im        …  0.212663-0.212663im   0.0+0.0im
 0.0690983-0.0690983im          0.0-0.0430396im  0.0+0.0im
  0.212663+0.212663im      0.132462+0.0im        0.0+0.0im
       0.0+0.0im                0.0+0.0im        0.0+0.0im

In [256]:
display(latexify(U))
display(latexify(rho_to_rec_; fmt="%.3f"))

L"\begin{equation}
\left[
\begin{array}{cccc}
1.0+0.0\mathit{i} & 0.0\mathit{i} & 0.0\mathit{i} & 0.0\mathit{i} \\
0.0\mathit{i} & 0.9510565162951536+0.0\mathit{i} & -0.30901699437494734\mathit{i} & 0.0\mathit{i} \\
0.0\mathit{i} & -0.3090169943749474\mathit{i} & 0.9510565162951535+0.0\mathit{i} & 0.0\mathit{i} \\
0.0\mathit{i} & 0.0\mathit{i} & 0.0\mathit{i} & 1.0+0.0\mathit{i} \\
\end{array}
\right]
\end{equation}
"

L"\begin{equation}
\left[
\begin{array}{cccc}
0.854+0.000\mathit{i} & 0.069+0.069\mathit{i} & 0.213-0.213\mathit{i} & 0.0\mathit{i} \\
0.069-0.069\mathit{i} & 0.014+0.000\mathit{i} & -0.043039578628754925\mathit{i} & 0.0\mathit{i} \\
0.213+0.213\mathit{i} & 0.043039578628754925\mathit{i} & 0.132+0.000\mathit{i} & 0.0\mathit{i} \\
0.0\mathit{i} & 0.0\mathit{i} & 0.0\mathit{i} & 0.0\mathit{i} \\
\end{array}
\right]
\end{equation}
"

### Step 2.2: Disentangling Noise
We apply the **real noise** only on the system. This reduces the shared correlations with the ancilla, and consequently the ''destructivness'' of the ancilla measurement

In [257]:
# This is equivalent to applying the supermap to the system alone,
#  extending the supermap to act as identity on the ancilla
rho_to_rec_ = apply_channel(config.real_noise.kraus, rho_to_rec_, n_qubits; extra_dims=size(η, 1))
rho1_ = apply_channel(state.noise_options[1].kraus, rho1_, n_qubits; extra_dims=size(η, 1))
rho2_ = apply_channel(state.noise_options[2].kraus, rho2_, n_qubits; extra_dims=size(η, 1))

4×4 Matrix{ComplexF64}:
  0.493008+0.0im        0.0345492+0.0345492im  …        0.0+0.0215198im
 0.0345492-0.0345492im  0.0069922+0.0im                 0.0+0.0im
  0.212663+0.0im              0.0+0.0215198im     0.0345492+0.0345492im
       0.0-0.0215198im        0.0+0.0im           0.0069922+0.0im

In [258]:
state.noise_options[2].kraus

2-element Vector{Matrix{ComplexF64}}:
 [0.7071067811865476 + 0.0im 0.0 + 0.0im; 0.0 + 0.0im 0.7071067811865476 + 0.0im]
 [0.0 + 0.0im 0.7071067811865476 + 0.0im; 0.7071067811865476 + 0.0im 0.0 + 0.0im]

In [259]:
display(latexify(rho_to_rec_; fmt="%.5f"))
display(latexify(rho2_; fmt="%.5f"))

L"\begin{equation}
\left[
\begin{array}{cccc}
0.91978+0.00000\mathit{i} & 0.06910+0.06910\mathit{i} & 0.15038-0.15038\mathit{i} & 0.0\mathit{i} \\
0.06910-0.06910\mathit{i} & 0.01398+0.00000\mathit{i} & -0.030433577907804217\mathit{i} & 0.0\mathit{i} \\
0.15038+0.15038\mathit{i} & 0.030433577907804217\mathit{i} & 0.06623+0.00000\mathit{i} & 0.0\mathit{i} \\
0.0\mathit{i} & 0.0\mathit{i} & 0.0\mathit{i} & 0.0\mathit{i} \\
\end{array}
\right]
\end{equation}
"

L"\begin{equation}
\left[
\begin{array}{cccc}
0.49301+0.00000\mathit{i} & 0.03455+0.03455\mathit{i} & 0.21266+0.00000\mathit{i} & 0.021519789314377466\mathit{i} \\
0.03455-0.03455\mathit{i} & 0.00699+0.00000\mathit{i} & -0.021519789314377466\mathit{i} & 0.0\mathit{i} \\
0.21266+0.00000\mathit{i} & 0.021519789314377466\mathit{i} & 0.49301+0.00000\mathit{i} & 0.03455+0.03455\mathit{i} \\
-0.021519789314377466\mathit{i} & 0.0\mathit{i} & 0.03455-0.03455\mathit{i} & 0.00699+0.00000\mathit{i} \\
\end{array}
\right]
\end{equation}
"

### Step 3.1: Measurement
We measure the output ancilla and compare it with the possible options, given the possible noises.
The measurement outcome may be 1 with probability `p1` or 2 with probability `p2`, where `p1` and `p2` are obtained for an optimal discrimination POVM.

In [260]:
η_test = ptrace_sys(rho_to_rec_, 2^n_qubits, 2)
η1 = ptrace_sys(rho1_, 2^n_qubits, 2)
η2 = ptrace_sys(rho2_, 2^n_qubits, 2)

# display(latexify(η_test; fmt="%.3f"))
# display(latexify(η1; fmt="%.6f"))
# display(latexify(η2; fmt="%.6f"))
display(η_test)
display(η1)
display(η2)

2×2 Matrix{ComplexF64}:
  0.986016+0.0im        0.0690983+0.0690983im
 0.0690983-0.0690983im  0.0139844+0.0im

2×2 Matrix{ComplexF64}:
  0.986016+0.0im        0.0690983+0.0690983im
 0.0690983-0.0690983im  0.0139844+0.0im

2×2 Matrix{ComplexF64}:
  0.986016+0.0im        0.0690983+0.0690983im
 0.0690983-0.0690983im  0.0139844+0.0im

In [261]:
function discrimin(ρ_test, ρ1, ρ2, ds, da, q1::Real = 0.5, q2::Real = 0.5; tol=1e-10)
  # Check this is actually a bipartite state of the expected dimensions
  size(ρ_test) == (ds*da, ds*da) || error("ρ_test has incompatible dimensions")
  size(ρ1) == (ds*da, ds*da) || error("ρ1 has incompatible dimensions")
  size(ρ2) == (ds*da, ds*da) || error("ρ2 has incompatible dimensions")
  
  # Trace out the system to get the reduced states of the ancilla
  η_test = ptrace_sys(ρ_test, ds, da)
  η1 = ptrace_sys(ρ1, ds, da)
  η2 = ptrace_sys(ρ2, ds, da)

  Δη = q1 * η1 - q2 * η2

  # Early exit: states are indistinguishable, return uniform
  if norm(Δη) < tol
      return [0.5, 0.5], [0.5*I(da), 0.5*I(da)]
  end

  eigen_decomp = eigen(Hermitian(Δη))
  eigenvalues  = eigen_decomp.values
  eigenvectors = eigen_decomp.vectors

  Π1 = zeros(ComplexF64, da, da)
  for i in eachindex(eigenvalues)
      if eigenvalues[i] > tol
          v   = eigenvectors[:, i]
          Π1 += v * v'
      end
  end
  Π2 = I(da) - Π1

  p1 = real(tr(Π1 * η_test))
  p2 = real(tr(Π2 * η_test))

  # Numerical sanity: p1 + p2 should be 1
  total = p1 + p2
  return [p1/total, p2/total], [Π1, Π2]
end

# ancillas must be normalized with the probabilities of their respective noise channels
q1 = state.noise_options[1].probability
q2 = state.noise_options[2].probability
w, Πs = discrimin(rho_to_rec_, rho1_, rho2_, 2^n_qubits, 2, q1, q2; tol=1e-12);
println("Probabilities of the POVM outputs: $w")

povm = sample(rng, [1, 2], Weights(w))
println("Measurement result: $povm")

Probabilities of the POVM outputs: [0.5, 0.5]
Measurement result: 1


In [262]:
display(Πs[1])
display(Πs[2])

2×2 Diagonal{Float64, Vector{Float64}}:
 0.5   ⋅ 
  ⋅   0.5

2×2 Diagonal{Float64, Vector{Float64}}:
 0.5   ⋅ 
  ⋅   0.5

In [263]:
display(latexify(Πs[1]; fmt="%.3f"))
display(latexify(Πs[2]; fmt="%.3f"))

L"\begin{equation}
\left[
\begin{array}{cc}
0.500 & 0.000 \\
0.000 & 0.500 \\
\end{array}
\right]
\end{equation}
"

L"\begin{equation}
\left[
\begin{array}{cc}
0.500 & 0.000 \\
0.000 & 0.500 \\
\end{array}
\right]
\end{equation}
"

### Step 3.2: Collapse
Given one measurement outcome on the ancilla, the system collapses in the relative state.

In [264]:
function collapse_state(ρ_SA, Π)
  da = size(Π, 1)
  ds = size(ρ_SA, 1) ÷ da
  # Measurement operator: M = I ⊗ Π
  M = kron(I(ds), Π)
  ρ_post = M * ρ_SA * M'
  Z = real(tr(ρ_post))  # Normalization factor
  # Regularization constant to avoid division by zero in pathological cases
  ε = 1e-12
  Z = max(Z, ε)
  return ρ_post / Z
end

collapsed_rho_to_rec = collapse_state(rho_to_rec_, Πs[povm])
collapsed_rho1 = collapse_state(rho1_, Πs[povm])
collapsed_rho2 = collapse_state(rho2_, Πs[povm])

rho_to_rec = ptrace_ancilla(collapsed_rho_to_rec, 2^n_qubits, size(η, 1))
rho1 = ptrace_ancilla(collapsed_rho1, 2^n_qubits, size(η, 1))
rho2 = ptrace_ancilla(collapsed_rho2, 2^n_qubits, size(η, 1))

2×2 Matrix{ComplexF64}:
      0.5+0.0im  0.212663+0.0im
 0.212663+0.0im       0.5+0.0im

In [265]:
display(latexify(collapsed_rho_to_rec; fmt="%.5f"))
display(latexify(rho_to_rec; fmt="%.5f"))

L"\begin{equation}
\left[
\begin{array}{cccc}
0.91978+0.00000\mathit{i} & 0.06910+0.06910\mathit{i} & 0.15038-0.15038\mathit{i} & 0.0\mathit{i} \\
0.06910-0.06910\mathit{i} & 0.01398+0.00000\mathit{i} & -0.030433577907804217\mathit{i} & 0.0\mathit{i} \\
0.15038+0.15038\mathit{i} & 0.030433577907804217\mathit{i} & 0.06623+0.00000\mathit{i} & 0.0\mathit{i} \\
0.0\mathit{i} & 0.0\mathit{i} & 0.0\mathit{i} & 0.0\mathit{i} \\
\end{array}
\right]
\end{equation}
"

L"\begin{equation}
\left[
\begin{array}{cc}
0.93377+0.00000\mathit{i} & 0.15038-0.15038\mathit{i} \\
0.15038+0.15038\mathit{i} & 0.06623+0.00000\mathit{i} \\
\end{array}
\right]
\end{equation}
"

In [266]:
display(latexify(rho_to_rec_; fmt="%.3f"))
display(latexify(rho1_; fmt="%.3f"))
display(latexify(rho2_; fmt="%.3f"))

display(latexify(collapsed_rho_to_rec; fmt="%.5f"))
display(latexify(collapsed_rho1; fmt="%.5f"))
display(latexify(collapsed_rho2; fmt="%.5f"))

display(latexify(rho_to_rec; fmt="%.3f"))
display(latexify(rho1; fmt="%.3f"))
display(latexify(rho2; fmt="%.3f"))

L"\begin{equation}
\left[
\begin{array}{cccc}
0.920+0.000\mathit{i} & 0.069+0.069\mathit{i} & 0.150-0.150\mathit{i} & 0.0\mathit{i} \\
0.069-0.069\mathit{i} & 0.014+0.000\mathit{i} & -0.030433577907804217\mathit{i} & 0.0\mathit{i} \\
0.150+0.150\mathit{i} & 0.030433577907804217\mathit{i} & 0.066+0.000\mathit{i} & 0.0\mathit{i} \\
0.0\mathit{i} & 0.0\mathit{i} & 0.0\mathit{i} & 0.0\mathit{i} \\
\end{array}
\right]
\end{equation}
"

L"\begin{equation}
\left[
\begin{array}{cccc}
0.920+0.000\mathit{i} & 0.069+0.069\mathit{i} & 0.150-0.150\mathit{i} & 0.0\mathit{i} \\
0.069-0.069\mathit{i} & 0.014+0.000\mathit{i} & -0.030433577907804217\mathit{i} & 0.0\mathit{i} \\
0.150+0.150\mathit{i} & 0.030433577907804217\mathit{i} & 0.066+0.000\mathit{i} & 0.0\mathit{i} \\
0.0\mathit{i} & 0.0\mathit{i} & 0.0\mathit{i} & 0.0\mathit{i} \\
\end{array}
\right]
\end{equation}
"

L"\begin{equation}
\left[
\begin{array}{cccc}
0.493+0.000\mathit{i} & 0.035+0.035\mathit{i} & 0.213+0.000\mathit{i} & 0.021519789314377466\mathit{i} \\
0.035-0.035\mathit{i} & 0.007+0.000\mathit{i} & -0.021519789314377466\mathit{i} & 0.0\mathit{i} \\
0.213+0.000\mathit{i} & 0.021519789314377466\mathit{i} & 0.493+0.000\mathit{i} & 0.035+0.035\mathit{i} \\
-0.021519789314377466\mathit{i} & 0.0\mathit{i} & 0.035-0.035\mathit{i} & 0.007+0.000\mathit{i} \\
\end{array}
\right]
\end{equation}
"

L"\begin{equation}
\left[
\begin{array}{cccc}
0.91978+0.00000\mathit{i} & 0.06910+0.06910\mathit{i} & 0.15038-0.15038\mathit{i} & 0.0\mathit{i} \\
0.06910-0.06910\mathit{i} & 0.01398+0.00000\mathit{i} & -0.030433577907804217\mathit{i} & 0.0\mathit{i} \\
0.15038+0.15038\mathit{i} & 0.030433577907804217\mathit{i} & 0.06623+0.00000\mathit{i} & 0.0\mathit{i} \\
0.0\mathit{i} & 0.0\mathit{i} & 0.0\mathit{i} & 0.0\mathit{i} \\
\end{array}
\right]
\end{equation}
"

L"\begin{equation}
\left[
\begin{array}{cccc}
0.91978+0.00000\mathit{i} & 0.06910+0.06910\mathit{i} & 0.15038-0.15038\mathit{i} & 0.0\mathit{i} \\
0.06910-0.06910\mathit{i} & 0.01398+0.00000\mathit{i} & -0.030433577907804217\mathit{i} & 0.0\mathit{i} \\
0.15038+0.15038\mathit{i} & 0.030433577907804217\mathit{i} & 0.06623+0.00000\mathit{i} & 0.0\mathit{i} \\
0.0\mathit{i} & 0.0\mathit{i} & 0.0\mathit{i} & 0.0\mathit{i} \\
\end{array}
\right]
\end{equation}
"

L"\begin{equation}
\left[
\begin{array}{cccc}
0.49301+0.00000\mathit{i} & 0.03455+0.03455\mathit{i} & 0.21266+0.00000\mathit{i} & 0.021519789314377466\mathit{i} \\
0.03455-0.03455\mathit{i} & 0.00699+0.00000\mathit{i} & -0.021519789314377466\mathit{i} & 0.0\mathit{i} \\
0.21266+0.00000\mathit{i} & 0.021519789314377466\mathit{i} & 0.49301+0.00000\mathit{i} & 0.03455+0.03455\mathit{i} \\
-0.021519789314377466\mathit{i} & 0.0\mathit{i} & 0.03455-0.03455\mathit{i} & 0.00699+0.00000\mathit{i} \\
\end{array}
\right]
\end{equation}
"

L"\begin{equation}
\left[
\begin{array}{cc}
0.934+0.000\mathit{i} & 0.150-0.150\mathit{i} \\
0.150+0.150\mathit{i} & 0.066+0.000\mathit{i} \\
\end{array}
\right]
\end{equation}
"

L"\begin{equation}
\left[
\begin{array}{cc}
0.934+0.000\mathit{i} & 0.150-0.150\mathit{i} \\
0.150+0.150\mathit{i} & 0.066+0.000\mathit{i} \\
\end{array}
\right]
\end{equation}
"

L"\begin{equation}
\left[
\begin{array}{cc}
0.500+0.000\mathit{i} & 0.213+0.000\mathit{i} \\
0.213+0.000\mathit{i} & 0.500+0.000\mathit{i} \\
\end{array}
\right]
\end{equation}
"

### Step 3.3: Complete the Map
To be able to recover the state from this perturbation, we create a CPTP map such that its output is the same as the measurement collapse.

In [267]:
function create_transfer_kraus(d, p, q)
  ks = Matrix{ComplexF64}[]
  if d == 2
    p1, p2 = p
    q1, q2 = q

    # Initialize trivial operators for edge cases
    k0 = Matrix{ComplexF64}(I, 2, 2)
    k1 = zeros(ComplexF64, 2, 2)

    if q1 < p1
      k0 = [sqrt(q1/p1) 0; 0 1]
      k1 = [0 0; sqrt(1-q1/p1) 0]
    elseif q2 > p2
      k0 = [1 0; 0 sqrt(p2/q2)]
      k1 = [0 sqrt(1-p2/q2); 0 0]
    end
    ks = [k0, k1]
  elseif d == 4
    alpha = min(p[1]*q[1], p[2]*q[2])
    beta = min(p[3]*q[3], p[4]*q[4])
    # Decrease alpha and beta
    alpha = alpha / 10.0
    beta = beta / 10.0
    @debug "ps: $(p), qs: $(q), alpha: $(alpha), beta: $(beta)"
    # K0: Main diagonal (k=0)
    K0 = diagm(0 => [
        sqrt((p[1]*q[1] - alpha) / p[1]),
        sqrt((p[2]*q[2] - alpha) / p[2]),
        sqrt((p[3]*q[3] - beta) / p[3]),
        sqrt((p[4]*q[4] - beta) / p[4])
    ])

    # K1: First super-diagonal (k=1)
    K1 = diagm(1 => [
        sqrt((p[2]*q[1] + alpha) / p[2]),
        sqrt((p[3]*q[2]) / p[3]),
        sqrt((p[4]*q[3] + beta) / p[4])
    ])

    # K2: First sub-diagonal (k=-1)
    K2 = diagm(-1 => [
        sqrt((p[1]*q[2] + alpha) / p[1]),
        sqrt((p[2]*q[3]) / p[2]),
        sqrt((p[3]*q[4] + beta) / p[3])
    ])

    # K3: Second super-diagonal (k=2)
    K3 = diagm(2 => [
        sqrt((p[3]*q[1]) / p[3]),
        sqrt((p[4]*q[2]) / p[4])
    ])

    # K4: Second sub-diagonal (k=-2)
    K4 = diagm(-2 => [
        sqrt((p[1]*q[3]) / p[1]),
        sqrt((p[2]*q[4]) / p[2])
    ])

    # K5: Third super-diagonal (k=3)
    K5 = diagm(3 => [
        sqrt((p[4]*q[1]) / p[4])
    ])

    # K6: Third sub-diagonal (k=-3)
    K6 = diagm(-3 => [
        sqrt((p[1]*q[4]) / p[1])
    ])

    ks = [K0, K1, K2, K3, K4, K5, K6]

  end

  return ks
end

function collapse_map(input_state, output_state)
  d = size(input_state, 1)

  function clean_eigenvalues(eigvals, tol=1e-10)
    cleaned = similar(eigvals)
    for i in eachindex(eigvals)
        val = real(eigvals[i])
        if abs(val) < tol
            cleaned[i] = 0.0
        elseif val < 0
            cleaned[i] = 0.0
        else
            cleaned[i] = val
        end
    end
    return cleaned
  end
  
  function stochastic_projection(states_in, states_out)
    # projectors = [states_out[:, i] * states_in[:, i]' for i in 1:length(states_in)]
    # return sum(projectors)
    return states_out * states_in'
  end

  eigen_decomp = eigen(Hermitian(input_state), sortby = x -> -real(x))
  p            = clean_eigenvalues(eigen_decomp.values)
  phi_in       = eigen_decomp.vectors

  eigen_decomp = eigen(Hermitian(output_state), sortby = x -> -real(x))
  q            = clean_eigenvalues(eigen_decomp.values)
  phi_out      = eigen_decomp.vectors

  basis = Matrix{ComplexF64}(I, d, d)

  U1 = stochastic_projection(phi_in, basis)
  superopU1 = kron(conj(U1), U1)

  K_transfer = create_transfer_kraus(d, p, q)
  superopK = kraus_to_superop(K_transfer)

  U2 = stochastic_projection(basis, phi_out)
  superopU2 = kron(conj(U2), U2)

  return superopU2 * superopK * superopU1
end

Cx = collapse_map(ptrace_ancilla(rho_to_rec_, 2^n_qubits, size(η, 1)), rho_to_rec)
C1 = collapse_map(ptrace_ancilla(rho1_, 2^n_qubits, size(η, 1)), rho1)
C2 = collapse_map(ptrace_ancilla(rho2_, 2^n_qubits, size(η, 1)), rho2)

4×4 Matrix{ComplexF64}:
         1.0+0.0im  -1.2326e-32+0.0im  -1.2326e-32+0.0im   1.2326e-32+0.0im
 -1.2326e-32+0.0im          1.0+0.0im   1.2326e-32+0.0im  -1.2326e-32+0.0im
 -1.2326e-32+0.0im   1.2326e-32+0.0im          1.0+0.0im  -1.2326e-32+0.0im
  1.2326e-32+0.0im  -1.2326e-32+0.0im  -1.2326e-32+0.0im          1.0+0.0im

In [268]:
# Proof
before_collapse = ptrace_ancilla(rho_to_rec_, 2^n_qubits, size(η, 1))
display(before_collapse)
after_collapse = rho_to_rec

state_to_check = unvec(Cx * vec(before_collapse))
norm(state_to_check - after_collapse) < 1e-12

2×2 Matrix{ComplexF64}:
 0.933769+0.0im        0.150375-0.150375im
 0.150375+0.150375im  0.0662311+0.0im

true

In [269]:
display(C2)

4×4 Matrix{ComplexF64}:
         1.0+0.0im  -1.2326e-32+0.0im  -1.2326e-32+0.0im   1.2326e-32+0.0im
 -1.2326e-32+0.0im          1.0+0.0im   1.2326e-32+0.0im  -1.2326e-32+0.0im
 -1.2326e-32+0.0im   1.2326e-32+0.0im          1.0+0.0im  -1.2326e-32+0.0im
  1.2326e-32+0.0im  -1.2326e-32+0.0im  -1.2326e-32+0.0im          1.0+0.0im

### Step 3.4: Update the Noise map
To recover all the previous perturbations, we must transform all the maps to one combined supermap,
$$
\mathcal{N} = \Lambda_{\text{complt}} \circ \Xi \circ \mathcal{N}\,,
$$
with 
$$
\Xi[\rho] = \text{Tr}_A\left[ (\Omega\otimes\mathbb{I}) U_{SA}(\rho\otimes\eta)U_{SA}^\dagger \right]
$$
This can be done composing the Kraus map for the noise and that for a system+ancilla unitary:
$$
\Omega[X]=\sum_m M_m X M^\dagger_m\,
$$
and, diagonalizing the ancilla as $\sum p_n|n\rangle \langle n|$,
$$
K_{i,n}​=\sqrt{p_n}​(\mathbb{I}_S\otimes\langle i|)U(\mathbb{I}_S\otimes|n\rangle)\,,
$$
so that the effective Kraus operators become:
$$
K_{m,i,n} = M_m\sqrt{p_n}​(\mathbb{I}_S\otimes\langle i|)U(\mathbb{I}_S\otimes|n\rangle)\,,
$$
and finally the supermap:
$$
S_\Xi=\sum_{m,i,n} = K^*_{m,i,n}\otimes K_{m,i,n}
$$

In [270]:
model.kraus_fwd[1]
config.real_noise.kraus[1]

2×2 Matrix{ComplexF64}:
 1.0+0.0im       0.0+0.0im
 0.0+0.0im  0.707107+0.0im

In [271]:
function compose_kraus(
    kraus2::Vector{<:AbstractMatrix},
    kraus1::Vector{<:AbstractMatrix},
)
    # channel 1 first, then channel 2
    out = Matrix{eltype(kraus1[1])}[]
    for K2 in kraus2
        for K1 in kraus1
            push!(out, K2 * K1)
        end
    end
    return out
end

real_system_kraus = config.real_noise.extended_kraus
option_system_kraus = [noise.extended_kraus for noise in state.noise_options]

Xi = kraus_to_superop(
    compose_kraus(real_system_kraus, model.kraus_fwd))
Xi1 = kraus_to_superop(
    compose_kraus(option_system_kraus[1], model.kraus_fwd))
Xi2 = kraus_to_superop(
    compose_kraus(option_system_kraus[2], model.kraus_fwd))


4×4 Matrix{ComplexF64}:
 0.5+0.0im       0.0+0.0im       0.0+0.0im  0.5+0.0im
 0.0+0.0im  0.475528+0.0im  0.475528+0.0im  0.0+0.0im
 0.0+0.0im  0.475528+0.0im  0.475528+0.0im  0.0+0.0im
 0.5+0.0im       0.0+0.0im       0.0+0.0im  0.5+0.0im

In [272]:
# Check that this supermap reproduces the collision + noise
before_collapse = ptrace_ancilla(rho_to_rec_, 2^n_qubits, size(η, 1))
isapprox(unvec(Xi * vec(state.ρ0)), before_collapse)

true

### Step 4: Choose the Noise
Based on the information extracted with the informative collision, choose a supermap to recover among `N1` and `N2`.

In [273]:
function update_noise_guess!(state, povm; inertia::Int=1)

  # update the count of noise choices
  if povm == 1
    append!(state.choice.c1, 1)
    append!(state.choice.c2, 0)
    state.choice.c1_count += 1
  elseif povm == 2
    append!(state.choice.c1, 0)
    append!(state.choice.c2, 1)
    state.choice.c2_count += 1
  end

  # Always choose the leading accumulated povm
  if state.choice.c1_count > state.choice.c2_count
    state.choice.current = 1
  elseif state.choice.c2_count > state.choice.c1_count
    state.choice.current = 2
  else
    # Look at the previous choices to break ties
    state.choice.current = state.choice.history[end]
  end

  append!(state.choice.history, state.choice.current)
end

update_noise_guess!(state, povm)
new_guess = state.noise_options[state.choice.current]
println("Updated noise guess:\t$(new_guess.name)")

Updated noise guess:	bitflip


### Step 5: Recovery

In [274]:
state.choice.current = 2

2

In [275]:
model = CollisionModel(state.choice.current == 1 ? C1 * Xi1 * N1 : C2 * Xi2 * N2, config.sigma)
P = kraus_to_superop(model.kraus_rec);

# Get the state state recovered by the Stinespring dilation of the Petz map
rho_rec, _ = apply_collision(model, rho_to_rec; trace=true)
model1 = CollisionModel(C1 * Xi1 * N1, config.sigma)
rho1, _ = apply_collision(model1, rho1; trace=true)
model2 = CollisionModel(C2 * Xi2 * N2, config.sigma)
rho2, _ = apply_collision(model2, rho2; trace=true)
;

In [276]:
kraus_to_superop(model.kraus_rec)

4×4 Matrix{ComplexF64}:
 0.872313+0.0im       -0.0441072-1.38778e-17im  …  0.872313+0.0im
 0.157388+0.235472im     0.15569-0.0278958im       0.157388+0.235472im
 0.157388-0.235472im     0.15569+0.0278958im       0.157388-0.235472im
 0.127687+0.0im        0.0441072+0.0im             0.127687+0.0im

In [277]:
println("\n - Collapse")
display(latexify(Cx))

println("\n - Collision + Noise")
display(latexify(Xi))

println("\n - Recovered State")
display(latexify(rho_rec; fmt="%.8f"))


 - Collapse


"\\begin{equation}\n\\left[\n\\begin{array}{cccc}\n0.9999999999999991+0.0\\mathit{i} & 9.609461774468023e-18-1.718383208772329e-17\\mathit{i} & 1.134418525044483e-17+1.544910861174648e-17\\mathit{i} & -4.5117097104798096e-18+0.0\\mathit{i} \\\\\n9.609461774468023e-18+1.718383208772329e-1" ⋯ 317 bytes ⋯ "69303768665e-17-1.1388087964663316e-17\\mathit{i} \\\\\n-4.5117097104798096e-18+0.0\\mathit{i} & 3.630969303768665e-17-3.914366358029223e-17\\mathit{i} & 3.630969303768665e-17+1.1388087964663316e-17\\mathit{i} & 0.9999999999999999+0.0\\mathit{i} \\\\\n\\end{array}\n\\right]\n\\end{equation}\n"


 - Collision + Noise


L"\begin{equation}
\left[
\begin{array}{cccc}
1.0+0.0\mathit{i} & 0.0\mathit{i} & 0.0\mathit{i} & 0.5477457514062631+0.0\mathit{i} \\
0.0\mathit{i} & 0.6724985119639574+0.0\mathit{i} & 0.0\mathit{i} & 0.0\mathit{i} \\
0.0\mathit{i} & 0.0\mathit{i} & 0.6724985119639574+0.0\mathit{i} & 0.0\mathit{i} \\
0.0\mathit{i} & 0.0\mathit{i} & 0.0\mathit{i} & 0.4522542485937369+0.0\mathit{i} \\
\end{array}
\right]
\end{equation}
"


 - Recovered State


L"\begin{equation}
\left[
\begin{array}{cc}
0.85904804+0.00000000\mathit{i} & 0.20421167-0.22708192\mathit{i} \\
0.20421167+0.22708192\mathit{i} & 0.14095196-0.00000000\mathit{i} \\
\end{array}
\right]
\end{equation}
"

### Step 6: Measure Fidelity

In [278]:
fid_initial = fidelity(state.ρ0, rho_rec)
fid_track = fidelity(state.ρ0, rho_free)

println("Fidelity after recovery: ", fid_initial)
println("Fidelity without recovery: ", fid_track)

Fidelity after recovery: 0.9995064420699636
Fidelity without recovery: 0.9999999999999964


---

## Iterate
Update the complete map. 
This requires using the supermap operators and multiplying them

In [279]:
function iterate_recovery!(
  step::Int, state::RecoveryState, config::RecoveryConfig, logs::RecoveryLogs
  )
  
  # Rename variables for readability
  Ox = config.real_noise.supermap_noise
  O1 = state.noise_options[1].supermap_noise
  O2 = state.noise_options[2].supermap_noise
  if step == 1
    Nx = I(size(config.real_noise.supermap)[1])
    N1 = I(size(config.real_noise.supermap)[1])
    N2 = I(size(config.real_noise.supermap)[1])
  else
    Nx = config.real_noise.supermap
    N1 = state.noise_options[1].supermap
    N2 = state.noise_options[2].supermap
  end
  
  ds = 2^config.n_qubits
  da = size(config.ancilla_state, 1)

  # ======================================
  # Step 1: Apply noise only on the reference state to track the fidelity without recovery
  rho_free = unvec((Ox)^step * vec(state.ρ0))
  
  rho_to_rec = unvec(Nx * vec(state.ρ0))
  rho1 = unvec(N1 * vec(state.ρ0))
  rho2 = unvec(N2 * vec(state.ρ0))

  # ======================================
  # Step 2: Informative collision
  η = config.ancilla_state
  U = config.collision_unitary
  model = CollisionModel(
      U, config.sigma, ds, da, ancilla_state=η)

  # Get the composite state system+ancilla
  rho_to_rec_ = apply_collision(model, rho_to_rec; ancilla_state=η, trace=false)
  rho1_ = apply_collision(model, rho1; ancilla_state=η, trace=false)
  rho2_ = apply_collision(model, rho2; ancilla_state=η, trace=false)
  # Apply noise only to the system
  rho_to_rec_ = apply_extended_channel(rho_to_rec_, config.real_noise.extended_kraus, ds)
  rho1_ = apply_extended_channel(rho1_, state.noise_options[1].extended_kraus, ds)
  rho2_ = apply_extended_channel(rho2_, state.noise_options[2].extended_kraus, ds)
  # Create the supermap corresponding to the collision followed by the noise
  Xi = kraus_to_superop(
    compose_kraus(config.real_noise.extended_kraus, model.kraus_fwd))
  Xi1 = kraus_to_superop(
    compose_kraus(state.noise_options[1].extended_kraus, model.kraus_fwd))
  Xi2 = kraus_to_superop(
    compose_kraus(state.noise_options[2].extended_kraus, model.kraus_fwd))

  # ======================================
  # Step 3: Measurement
  q1 = state.noise_options[1].probability
  q2 = state.noise_options[2].probability
  w, Πs = discrimin(rho_to_rec_, rho1_, rho2_, ds, da, q1, q2);
  @debug "Probabilities of the POVM outputs: $w"

  povm = sample(config.rng, [1, 2], Weights(w))
  @debug "Measurement result: $povm"

  # Collapse the system and trace out the ancilla
  rho_to_rec = ptrace_ancilla(collapse_state(rho_to_rec_, Πs[povm]), ds, da)
  rho1 = ptrace_ancilla(collapse_state(rho1_, Πs[povm]), ds, da)
  rho2 = ptrace_ancilla(collapse_state(rho2_, Πs[povm]), ds, da)
  # Get the CPTP map corresponding to the collapse
  Cx = collapse_map(ptrace_ancilla(rho_to_rec_, ds, da), rho_to_rec)
  C1 = collapse_map(ptrace_ancilla(rho1_, ds, da), rho1)
  C2 = collapse_map(ptrace_ancilla(rho2_, ds, da), rho2)

  # Update the supermaps in the config and state
  if step == 1
    Nx = Cx * Xi
    N1 = C1 * Xi1
    N2 = C2 * Xi2
  else
    Nx = Cx * Xi * Nx
    N1 = C1 * Xi1 * N1
    N2 = C2 * Xi2 * N2
  end

  # ======================================
  # Step 4: Update noise guess and recovery map
  update_noise_guess!(state, povm)
  new_guess = state.noise_options[state.choice.current]
  @debug "Updated choice: " Ω=new_guess.name
  model = CollisionModel(state.choice.current == 1 ? N1 : N2, config.sigma)
  P = kraus_to_superop(model.kraus_rec)

  # ======================================
  # Step 5: Recovery
  rho_rec, _ = apply_collision(model, rho_to_rec; trace=true)
  model1 = CollisionModel(N1, config.sigma)
  rho1, _ = apply_collision(model1, rho1; trace=true)
  model2 = CollisionModel(N2, config.sigma)
  rho2, _ = apply_collision(model2, rho2; trace=true)
  
  fid_initial = fidelity(state.ρ0, rho_rec)
  fid_track = fidelity(state.ρ0, rho_free)

  @debug "Fidelity wrt initial state: " fid_initial
  @debug "Fidelity of free evolution: " fid_track

  # ======================================
  # Step 6: Updates for the next iteration
  # Save superoperators at current step
  push!(
    logs.maps, 
    (Nx=copy(Nx), N1=copy(N1), N2=copy(N2), P=copy(P), Cx=copy(Cx), Xi=copy(Xi))
    )
  push!(logs.ref_fidelities, fid_track)
  push!(logs.fidelities, fid_initial)
  config.real_noise.supermap = P * Nx
  state.noise_options[1].supermap = P * N1
  state.noise_options[2].supermap = P * N2

end

iterate_recovery! (generic function with 1 method)

In [ ]:
# cfg, st, lgs = deepcopy(config), deepcopy(state), deepcopy(logs)

cfg, st, lgs = load_configuration("../configs/config.toml");

total_steps = cfg.n_timesteps
# with_logger(debuglogger) do
@debug "Starting state: " cfg.sigma
for step in 1:total_steps
  iterate_recovery!(step, st, cfg, lgs)

  step == 1 && @printf("| Step\t| No Recovery\t| With Recovery |\n")
  # @printf("| %d\t|", step)
  # @printf(" %.6f\t|", lgs.ref_fidelities[end])
  # @printf(" %.6f\t|\n", lgs.fidelities[end])
end
# end

display(latexify(cfg.sigma; fmt="%.3f"))
display(latexify(st.ρ0; fmt="%.3f"))
g1 = plot_autorecovery(
  st, cfg, lgs; 
  xlims=[0, total_steps],
  ylims=[0.9, 1.01],
  show=false
  )
g2 = plot(
  [cumsum(st.choice.c1), cumsum(st.choice.c2)], 
  label=[st.noise_options[1].name st.noise_options[2].name]
  )

plot(g1, g2, layout=grid(2, 1), size=(1200, 550))

Using specified real noise from config: random



LoadError: ArgumentError: Noise type random requires a RNG.

In [ ]:
st.noise_options[1].name
real_noise_idx

In [ ]:
g1 = plot_autorecovery(
  st, cfg, lgs; 
  xlims=[0, 30],
  ylims=[0.98, 1.01],
  show=false,
  choices=false
)

output_folder = joinpath(cfg.experiment_dir, "visualization")
println(output_folder)
noise_labels = [
  titlecase(join(split(noise.name, '_'), ' ')) for noise in st.noise_options 
]
real_noise_idx = findfirst(n -> n.name == cfg.real_noise.name, st.noise_options)
for i in eachindex(noise_labels)
  if i == real_noise_idx
    noise_labels[i] *= " ☒"
  else
    noise_labels[i] *= " ☐"
  end
end

g2 = plot!(title="{$(noise_labels[1]) / $(noise_labels[2])} Adaptive Recovery")
savefig(g2, joinpath(output_folder, "adaptive_recovery.png"))
savefig(g2, joinpath(output_folder, "adaptive_recovery.pdf"))
g2

In [ ]:
function save_results!(cfg::RecoveryConfig, state::RecoveryState, logs::RecoveryLogs, avg_fidelities)

  function _serialize_maps(logs::RecoveryLogs)
    return [
        Dict(
            "Nx" => vec(real(m.Nx)),
            "N1" => vec(real(m.N1)),
            "N2" => vec(real(m.N2)),
            "P" => vec(real(m.P)),
            "dim" => size(m.Nx)
        ) for m in logs.maps
    ]
  end

  out_file = joinpath(cfg.experiment_dir, "data", "results.json")
  payload = Dict(
      "timestamp" => string(now()),
      "experiment" => cfg.name,
      "n_qubits" => cfg.n_qubits,
      "n_timesteps" => cfg.n_timesteps,
      "n_states" => cfg.n_states,
      "seed" => cfg.seed,
      "real_noise" => cfg.real_noise.name,
      "noise_options" => [n.name for n in state.noise_options],
      "fidelities" => logs.fidelities,
      "ref_fidelities" => logs.ref_fidelities,
      "choice_history" => state.choice.history,
      "choice_c1" => state.choice.c1,
      "choice_c2" => state.choice.c2,
      "avg_ref_fidelities" => avg_fidelities[1],
      "avg_fidelities" => avg_fidelities[2],
      "maps" => _serialize_maps(logs)
  )

  open(out_file, "w") do io
      JSON.print(io, payload, 2)
  end
  return out_file
end

save_results!(cfg, st, lgs, (lgs.ref_fidelities, lgs.fidelities))

In [ ]:
include("../julia/src/DecoKiller.jl")
using .DecoKiller

In [ ]:
cfg, state, logs, avg_fidelities = DecoKiller.run_experiment("../configs/config.toml")

In [ ]:
logs.fidelities